# Boulder Crop Export (run on Sherlock)

Exports image crops + model orientation measurements for manual annotation.
Run this on Sherlock, then `scp` the output directory to your local machine.

**Output:** `~/tmp/boulder_annotation_crops/`
- `crops/XXXXX.png` — grayscale image crop for each boulder
- `overlays/XXXXX.png` — same crop with all model mask outlines drawn
- `metadata.csv` — boulder ID, area, and orientation from each model

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.mask
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import box as shapely_box
from shapely import segmentize
from tqdm.auto import tqdm

from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

print("Imports OK")

In [ ]:
IN_RASTER   = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")
work_dir    = Path.home() / "tmp" / "YOLOv8BeyondEarth"

PRED_SHPS = {
    "yolo":     sorted((work_dir / "exp_yolo_256").glob("*-downscaled-mask-nms.shp")),
    "sam2_zs":  sorted((work_dir / "exp_sam2_256").glob("*-downscaled-mask-nms.shp")),
    "sam2_ft":  sorted((work_dir / "exp_sam2_finetuned_256").glob("*-downscaled-mask-nms.shp")),
    "sam2_auto":sorted((work_dir / "exp_sam2_auto_256").glob("*-mask-nms.shp")),
}

OUT_DIR   = Path.home() / "tmp" / "boulder_annotation_crops"
CROP_DIR  = OUT_DIR / "crops"
OVL_DIR   = OUT_DIR / "overlays"
for d in [CROP_DIR, OVL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

N_EXPORT  = 100    # number of boulders to export
PAD_FRAC  = 0.6    # padding around boulder bbox as fraction of boulder size
CROP_SIZE = 256    # output PNG size in pixels
AR_MIN, AR_MAX = 1.2, 2.0
MIN_AREA_M2    = 5.0

MODEL_COLORS = {
    "yolo":     (100, 180, 255),   # blue
    "sam2_zs":  (255, 100, 100),   # red
    "sam2_ft":  (255, 180,  50),   # orange
    "sam2_auto":(160, 100, 255),   # purple
}

for name, paths in PRED_SHPS.items():
    print(f"  {name}: {len(paths)} shapefiles")

In [ ]:
model_gdfs = {}
for name, paths in PRED_SHPS.items():
    if not paths:
        continue
    gdfs = [gpd.read_file(p) for p in paths]
    gdf  = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf["poly_area"] = gdf.geometry.area
    model_gdfs[name] = gdf
    print(f"  {name}: {len(gdf):,} detections")

# Use SAM2 fine-tuned as the source model for sampling boulders
SOURCE = "sam2_ft"
gdf_source = model_gdfs[SOURCE]

with rasterio.open(IN_RASTER) as src:
    RES = src.res[0]
print(f"Raster resolution: {RES:.4f} m/px")

In [ ]:
def measure_orientation(geom, res=RES):
    if geom is None or geom.is_empty:
        return None
    try:
        row_seg = pd.Series({"geometry": segmentize(geom, res)})
        ellipse_poly, _, _, _ = fitEllipse(row_seg)
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
        if short_ax < 1e-6:
            return None
        return float(angle180 % 180), float(long_ax / short_ax)
    except Exception:
        return None


def find_match(target_geom, gdf, iou_thresh=0.25):
    """Find best-IoU matching detection in gdf for target_geom."""
    candidates = gdf[gdf.geometry.intersects(target_geom.buffer(2))]
    best_iou, best_row = 0.0, None
    for _, row in candidates.iterrows():
        try:
            inter = target_geom.intersection(row.geometry).area
            union = target_geom.union(row.geometry).area
            iou   = inter / union if union > 0 else 0.0
        except Exception:
            continue
        if iou > best_iou:
            best_iou, best_row = iou, row
    return best_row if best_iou >= iou_thresh else None


print("Utility functions defined")

In [ ]:
candidates = []
for idx, src_row in tqdm(gdf_source.iterrows(), total=len(gdf_source), desc="Scanning"):
    geom = src_row.geometry
    if geom is None or geom.is_empty:
        continue
    area = src_row.poly_area
    if area < MIN_AREA_M2:
        continue
    r = measure_orientation(geom)
    if r is None or not (AR_MIN <= r[1] <= AR_MAX):
        continue

    # Check how many other models have a matching detection
    n_models = 1  # source model itself
    for name, mgdf in model_gdfs.items():
        if name == SOURCE:
            continue
        if find_match(geom, mgdf) is not None:
            n_models += 1

    if n_models >= 2:   # at least source + one other model
        candidates.append({"idx": idx, "geom": geom, "area": area,
                            "angle": r[0], "ar": r[1], "n_models": n_models})

print(f"Found {len(candidates)} candidate boulders")

# Sample N_EXPORT spread evenly across area sizes
df_cand = pd.DataFrame([{k: v for k, v in c.items() if k != "geom"} for c in candidates])
df_cand["geom"] = [c["geom"] for c in candidates]
df_cand = df_cand.sort_values("area").reset_index(drop=True)
step    = max(1, len(df_cand) // N_EXPORT)
sampled = df_cand.iloc[::step].head(N_EXPORT).reset_index(drop=True)
print(f"Sampled {len(sampled)} boulders (area range: {sampled.area.min():.1f}–{sampled.area.max():.1f} m²)")

In [ ]:
def geom_to_px(geom, crop_bounds, crop_hw):
    """Convert geo polygon → pixel coords in the crop image."""
    minx_c, miny_c, maxx_c, maxy_c = crop_bounds
    h, w = crop_hw
    sx = w / (maxx_c - minx_c)
    sy = h / (maxy_c - miny_c)
    pts = np.array(geom.exterior.coords[:-1])
    px  = ((pts[:, 0] - minx_c) * sx).astype(np.int32)
    py  = (h - (pts[:, 1] - miny_c) * sy).astype(np.int32)   # flip y
    return np.stack([px, py], axis=-1)


def draw_orientation_line(img_rgb, angle180, cx_px, cy_px, color, length_frac=0.38):
    """Draw orientation line through centroid. angle180 in geospatial convention."""
    h, w = img_rgb.shape[:2]
    L    = min(h, w) * length_frac
    # geo angle: 0=North(up), CW. In image: y-down, so North=up means -y direction.
    rad  = np.radians(angle180)
    dx   = np.sin(rad) * L    # East component
    dy   = -np.cos(rad) * L   # North component (negative because y-down)
    p1   = (int(cx_px - dx), int(cy_px - dy))
    p2   = (int(cx_px + dx), int(cy_px + dy))
    cv2.line(img_rgb, p1, p2, color, 2, cv2.LINE_AA)


records = []

with rasterio.open(IN_RASTER) as src:
    for i, row in tqdm(sampled.iterrows(), total=len(sampled), desc="Exporting"):
        geom = row["geom"]
        bid  = f"{i:05d}"

        # Compute padded crop bounds
        minx, miny, maxx, maxy = geom.bounds
        bw = maxx - minx
        bh = maxy - miny
        pad = max(bw, bh) * PAD_FRAC
        crop_box = shapely_box(minx - pad, miny - pad, maxx + pad, maxy + pad)
        cb = crop_box.bounds  # (minx, miny, maxx, maxy)

        try:
            out_img, _ = rasterio.mask.mask(src, [crop_box], crop=True)
        except Exception:
            continue

        raw = out_img[0].astype(np.float32)
        if raw.max() > 0:
            raw = (raw / raw.max() * 255).clip(0, 255).astype(np.uint8)
        raw_resized = cv2.resize(raw, (CROP_SIZE, CROP_SIZE))
        cv2.imwrite(str(CROP_DIR / f"{bid}.png"), raw_resized)

        # Overlay: draw model mask outlines + orientation lines
        overlay = cv2.cvtColor(raw_resized, cv2.COLOR_GRAY2RGB)
        cx_px   = CROP_SIZE // 2   # source boulder is approximately centered
        cy_px   = CROP_SIZE // 2

        rec = {"boulder_id": bid, "area_m2": row["area"],
               f"{SOURCE}_angle": row["angle"], f"{SOURCE}_ar": row["ar"]}

        # Draw source model mask
        src_pts = geom_to_px(geom, cb, (CROP_SIZE, CROP_SIZE))
        src_pts_scaled = (src_pts * CROP_SIZE / max(raw.shape[0], 1)).astype(np.int32)  # already in crop px
        cv2.polylines(overlay, [geom_to_px(geom, cb, (CROP_SIZE, CROP_SIZE))],
                      True, MODEL_COLORS[SOURCE], 1)
        draw_orientation_line(overlay, row["angle"], cx_px, cy_px, MODEL_COLORS[SOURCE])

        for name, mgdf in model_gdfs.items():
            if name == SOURCE:
                continue
            match = find_match(geom, mgdf)
            if match is not None:
                r = measure_orientation(match.geometry)
                if r:
                    rec[f"{name}_angle"] = r[0]
                    rec[f"{name}_ar"]    = r[1]
                    # Draw mask outline
                    cv2.polylines(overlay,
                                  [geom_to_px(match.geometry, cb, (CROP_SIZE, CROP_SIZE))],
                                  True, MODEL_COLORS[name], 1)
                    draw_orientation_line(overlay, r[0], cx_px, cy_px, MODEL_COLORS[name])

        cv2.imwrite(str(OVL_DIR / f"{bid}.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
        records.append(rec)

df_meta = pd.DataFrame(records)
df_meta.to_csv(OUT_DIR / "metadata.csv", index=False)
print(f"\nExported {len(df_meta)} boulders → {OUT_DIR}")
print(df_meta.head())

In [ ]:
bids = df_meta["boulder_id"].tolist()[:8]
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for col_i, bid in enumerate(bids):
    axes[0, col_i].imshow(plt.imread(CROP_DIR / f"{bid}.png"), cmap="gray")
    axes[0, col_i].set_title(f"#{bid}", fontsize=7)
    axes[0, col_i].axis("off")
    axes[1, col_i].imshow(plt.imread(OVL_DIR / f"{bid}.png"))
    axes[1, col_i].axis("off")
axes[0, 0].set_ylabel("Crop", fontsize=8)
axes[1, 0].set_ylabel("Overlay", fontsize=8)
plt.suptitle("Sample exports — top: clean crop, bottom: mask outlines + model orientation lines",
             fontsize=10)
plt.tight_layout()
plt.show()

print(f"\nTo copy to local machine:")
print(f"  scp -r {OUT_DIR} cayleigh@sherlock.stanford.edu:{OUT_DIR} .")